# Did your team deserve better in the postseason?
Analysis behind the article. Two questions, each on actual records and on schedule-adjusted records:

1. **Did your team deserve to make the postseason?** Teams that missed despite a top-12 record in MLB, compared with the worst team that made it.
2. **Did your team deserve a better seed?** Wild Cards with a better record than the worst division winner in their league.

**Win difference, in games:** the way the standings measure it, ((W − L of team A) − (W − L of team B)) / 2. For schedule-adjusted records: the difference in schedule-adjusted win % × 162.

**Schedule-adjusted record:** the record a team would have had playing every other team equally often on neutral fields, estimated from every game it played that season (`src/ratings.py`).

The published charts cover the 12-team format (2022–2026). Sections 1–3 also show 2013–2021 for context. 2020 is excluded (60 games, 16-team field).

Run `scripts/pull_standings.py`, `scripts/pull_postseason.py` and `scripts/pull_games.py` first to download the data.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, '..')
from src.ratings import fit_all

RAW = Path('../data/raw')
FORMATS = ['10-team format (2013–21)', '12-team format (2022–26)']

standings = pd.read_csv(RAW / 'standings_1998_2026.csv')
post = pd.read_csv(RAW / 'postseason_teams_1998_2026.csv')
games = pd.read_csv(RAW / 'games_1998_2026.csv')

in_window = lambda d: d[(d['season'] >= 2013) & (d['season'] != 2020)]
df = in_window(standings).copy()
df = df.merge(post[['season', 'team_id']].assign(played_postseason=True), on=['season', 'team_id'], how='left')
# Playoff field: played a postseason game, or clinched (catches 2026's bye teams, who haven't played yet)
df['made_playoffs'] = df['played_postseason'].fillna(False).astype(bool) | df['clinched']
df = df.merge(fit_all(in_window(games))[['season', 'team_id', 'adj_pct']], on=['season', 'team_id'])
df['team'] = df['team_id'].map(df[df['season'] == 2026].set_index('team_id')['team'])
df['net'] = df['wins'] - df['losses']
df['pct'] = df['wins'] / (df['wins'] + df['losses'])
df['format'] = np.where(df['season'] <= 2021, FORMATS[0], FORMATS[1])

LENSES = {
    # how to rank teams, how to label a record, how to compute a gap in games
    'actual': dict(key='net', pct='pct',
                   record=lambda t: f"{t['wins']}–{t['losses']}",
                   label=lambda t: f"{t['team']} {t['wins']}–{t['losses']}",
                   gap=lambda a, b: (a['net'] - b['net']) / 2),
    'adjusted': dict(key='adj_pct', pct='adj_pct',
                     record=lambda t: f"~{round(t['adj_pct'] * 162)}–{162 - round(t['adj_pct'] * 162)}",
                     label=lambda t: f"{t['team']} ~{round(t['adj_pct'] * 162)}–{162 - round(t['adj_pct'] * 162)}",
                     gap=lambda a, b: round((a['adj_pct'] - b['adj_pct']) * 162, 1)),
}

def compare(groups, upper_name, lower_name, pick_upper, pick_lower, lens):
    # For each group (a season or a league-season): the best team on the "should be higher" side
    # vs. the worst team on the "got the spot" side, and the gap between them
    L = LENSES[lens]
    rows = []
    for keys, d in groups:
        keys = keys if isinstance(keys, tuple) else (keys,)
        upper_pool, lower_pool = pick_upper(d), pick_lower(d)
        up = upper_pool.loc[upper_pool[L['key']].idxmax()]
        low = lower_pool.loc[lower_pool[L['key']].idxmin()]
        rows.append({'season': keys[0], 'league': keys[1] if len(keys) > 1 else 'both',
                     'format': d['format'].iloc[0],
                     upper_name: L['label'](up), lower_name: L['label'](low),
                     'gap (games)': L['gap'](up, low),
                     'teams on the wrong side': int((upper_pool[L['key']] > low[L['key']]).sum()),
                     '_up_pct': up[L['pct']], '_low_pct': low[L['pct']],
                     '_up_label': L['label'](up), '_low_label': L['label'](low),
                     '_up_team': up['team'], '_low_team': low['team'],
                     '_up_record': L['record'](up), '_low_record': L['record'](low)})
    return pd.DataFrame(rows)

missed = lambda d: d[~d['made_playoffs']]
made = lambda d: d[d['made_playoffs']]
wild_cards = lambda d: d[d['made_playoffs'] & (d['division_rank'] != 1)]
div_winners = lambda d: d[d['division_rank'] == 1]

tables = {}
for lens in LENSES:
    tables[('cutoff_league', lens)] = compare(df.groupby(['season', 'league']), 'best non-playoff team',
                                              'worst playoff team', missed, made, lens)
    tables[('cutoff_mlb', lens)] = compare(df.groupby('season'), 'best non-playoff team',
                                           'worst playoff team', missed, made, lens)
    tables[('wildcard', lens)] = compare(df.groupby(['season', 'league']), 'best Wild Card',
                                         'worst division winner', wild_cards, div_winners, lens)

def show(name, lens):
    t = tables[(name, lens)]
    return t[[c for c in t if not c.startswith('_')]]

def counts(name):
    out = []
    for lens in LENSES:
        t = tables[(name, lens)]
        g = t.groupby('format')
        out.append(pd.DataFrame({'lens': lens, 'out of': g.size(), 'gap > 0': g['gap (games)'].apply(lambda s: (s > 0).sum()),
                                 'tied': g['gap (games)'].apply(lambda s: (s == 0).sum()),
                                 'largest gap': g['gap (games)'].max()}))
    return pd.concat(out)

## 1. The playoff cutoff, within each league

In [2]:
counts('cutoff_league')

,lens,out of,gap > 0,tied,largest gap
format,,,,,
10-team format (2013–21),actual,16,0,0,-1.0
12-team format (2022–26),actual,10,1,3,1.0
10-team format (2013–21),adjusted,16,3,0,6.3
12-team format (2022–26),adjusted,10,4,0,2.4


In [3]:
show('cutoff_league', 'actual').query('`gap (games)` >= 0')

,season,league,format,best non-playoff team,worst playoff team,gap (games),teams on the wrong side
18,2023,AL,12-team format (2022–26),SEA 88–74,MIN 87–75,1.0,1
21,2024,NL,12-team format (2022–26),AZ 89–73,ATL 89–73,0.0,0
22,2025,AL,12-team format (2022–26),HOU 87–75,DET 87–75,0.0,0
23,2025,NL,12-team format (2022–26),NYM 83–79,CIN 83–79,0.0,0


Within its own league, the playoff field almost always takes the teams with the best records. On actual records there's **one** league-season where a team missed with a better record than a playoff team (2023 AL: Seattle 88–74, Minnesota 87–75 as AL Central champion), plus three teams that tied the last playoff team and lost the tiebreaker.

## 2. The playoff cutoff, across both leagues

In [4]:
counts('cutoff_mlb')

,lens,out of,gap > 0,tied,largest gap
format,,,,,
10-team format (2013–21),actual,8,4,1,4.0
12-team format (2022–26),actual,5,4,1,5.0
10-team format (2013–21),adjusted,8,5,0,12.2
12-team format (2022–26),adjusted,5,4,0,8.8


In [5]:
show('cutoff_mlb', 'actual')

,season,league,format,best non-playoff team,worst playoff team,gap (games),teams on the wrong side
0,2013,both,10-team format (2013–21),TEX 91–72,CIN 90–72,0.5,1
1,2014,both,10-team format (2013–21),SEA 87–75,ATH 88–74,-1.0,0
2,2015,both,10-team format (2013–21),LAA 85–77,HOU 86–76,-1.0,0
3,2016,both,10-team format (2013–21),DET 86–75,NYM 87–75,-0.5,0
4,2017,both,10-team format (2013–21),MIL 86–76,MIN 85–77,1.0,1
5,2018,both,10-team format (2013–21),TB 90–72,ATL 90–72,0.0,0
6,2019,both,10-team format (2013–21),CLE 93–69,MIL 89–73,4.0,1
7,2021,both,10-team format (2013–21),TOR 91–71,ATL 88–73,2.5,2
8,2022,both,12-team format (2022–26),MIL 86–76,TB 86–76,0.0,0
9,2023,both,12-team format (2022–26),SEA 88–74,MIA 84–78,4.0,1


## 3. Wild Card vs. division winner, within each league

In [6]:
counts('wildcard')

,lens,out of,gap > 0,tied,largest gap
format,,,,,
10-team format (2013–21),actual,16,7,0,17.5
12-team format (2022–26),actual,10,5,2,12.5
10-team format (2013–21),adjusted,16,10,0,20.2
12-team format (2022–26),adjusted,10,8,0,15.9


In [7]:
show('wildcard', 'actual')

,season,league,format,best Wild Card,worst division winner,gap (games),teams on the wrong side
0,2013,AL,10-team format (2013–21),CLE 92–70,DET 93–69,-1.0,0
1,2013,NL,10-team format (2013–21),PIT 94–68,LAD 92–70,2.0,1
2,2014,AL,10-team format (2013–21),KC 89–73,DET 90–72,-1.0,0
3,2014,NL,10-team format (2013–21),PIT 88–74,STL 90–72,-2.0,0
4,2015,AL,10-team format (2013–21),NYY 87–75,TEX 88–74,-1.0,0
5,2015,NL,10-team format (2013–21),PIT 98–64,NYM 90–72,8.0,2
6,2016,AL,10-team format (2013–21),TOR 89–73,BOS 93–69,-4.0,0
7,2016,NL,10-team format (2013–21),NYM 87–75,LAD 91–71,-4.0,0
8,2017,AL,10-team format (2013–21),NYY 91–71,BOS 93–69,-2.0,0
9,2017,NL,10-team format (2013–21),AZ 93–69,CHC 92–70,1.0,1


In [8]:
# Does adjusting for schedule make the Wild Card / division winner gap bigger or smaller?
a, j = tables[('wildcard', 'actual')], tables[('wildcard', 'adjusted')]
either = (a['gap (games)'] > 0) | (j['gap (games)'] > 0)
bigger = (j['gap (games)'] > a['gap (games)'])[either]
print(f'Of {either.sum()} league-seasons with a positive gap either way, the adjusted gap is bigger in {bigger.sum()}.')

Of 18 league-seasons with a positive gap either way, the adjusted gap is bigger in 16.


## 4. Every team that missed with a better record (12-team format)
The charts show only the largest gap each season. Here is every team that missed the postseason with a better record than at least one playoff team (either league), and every playoff team it was better than.

In [9]:
twelve = df[df['season'] >= 2022].copy()
twelve['record'] = twelve['wins'].astype(str) + '–' + twelve['losses'].astype(str)
rows = []
for season, d in twelve.groupby('season'):
    made_ = d[d['made_playoffs']]
    for _, t in d[~d['made_playoffs']].sort_values('net', ascending=False).iterrows():
        for _, p in made_[made_['net'] <= t['net']].sort_values('net').iterrows():
            rows.append({'season': season, 'missed': t['team'], 'missed record': t['record'], 'missed league': t['league'],
                         'playoff team': p['team'], 'playoff record': p['record'], 'playoff league': p['league'],
                         'gap (games)': (t['net'] - p['net']) / 2,
                         'tie': t['net'] == p['net']})
all_misses = pd.DataFrame(rows)
all_misses

,season,missed,missed record,missed league,playoff team,playoff record,playoff league,gap (games),tie
0,2022,MIL,86–76,NL,TB,86–76,AL,0.0,True
1,2023,SEA,88–74,AL,MIA,84–78,NL,4.0,False
2,2023,SEA,88–74,AL,AZ,84–78,NL,4.0,False
3,2023,SEA,88–74,AL,MIN,87–75,AL,1.0,False
4,2024,AZ,89–73,NL,KC,86–76,AL,3.0,False
5,2024,AZ,89–73,NL,DET,86–76,AL,3.0,False
6,2024,AZ,89–73,NL,HOU,88–73,AL,0.5,False
7,2024,AZ,89–73,NL,ATL,89–73,NL,0.0,True
8,2024,AZ,89–73,NL,NYM,89–73,NL,0.0,True
9,2025,HOU,87–75,AL,CIN,83–79,NL,4.0,False


## 4b. Every Wild Card with a better record than a division winner (12-team format)
The Wild Card charts show only the best Wild Card vs. the worst division winner. Here is every Wild Card that finished ahead of at least one division winner in its own league.

In [10]:
rows = []
for (season, league), d in twelve.groupby(['season', 'league']):
    dws = d[d['division_rank'] == 1]
    for _, w in d[d['made_playoffs'] & (d['division_rank'] != 1)].sort_values('net', ascending=False).iterrows():
        for _, v in dws[dws['net'] <= w['net']].sort_values('net').iterrows():
            rows.append({'season': season, 'league': league, 'wild card': w['team'], 'wild card record': w['record'],
                         'division winner': v['team'], 'division winner record': v['record'],
                         'gap (games)': (w['net'] - v['net']) / 2, 'tie': w['net'] == v['net']})
all_wildcards = pd.DataFrame(rows)
print('Wild Cards with a strictly better record than a division winner:',
      all_wildcards.loc[~all_wildcards['tie']].groupby(['season', 'wild card']).ngroups)
all_wildcards

Wild Cards with a strictly better record than a division winner: 10


,season,league,wild card,wild card record,division winner,division winner record,gap (games),tie
0,2022,AL,TOR,92–70,CLE,92–70,0.0,True
1,2022,NL,NYM,101–61,STL,93–69,8.0,False
2,2022,NL,NYM,101–61,ATL,101–61,0.0,True
3,2023,AL,TB,99–63,MIN,87–75,12.0,False
4,2023,AL,TB,99–63,HOU,90–72,9.0,False
5,2023,AL,TEX,90–72,MIN,87–75,3.0,False
6,2023,AL,TEX,90–72,HOU,90–72,0.0,True
7,2023,AL,TOR,89–73,MIN,87–75,2.0,False
8,2024,AL,BAL,91–71,HOU,88–73,2.5,False
9,2024,NL,SD,93–69,MIL,93–69,0.0,True


## 5. Seeds by record vs. actual seeds (12-team format)
What if each league's six playoff teams were seeded 1–6 purely by record, with no guaranteed top-three seeds for division winners? Ties keep MLB's tiebreaker order (`league_rank`). The top two seeds get byes.

In [11]:
rows = []
for (season, league), d in twelve[twelve['made_playoffs']].groupby(['season', 'league']):
    order = ['pct', 'league_rank']
    dw = d[d['division_rank'] == 1].sort_values(order, ascending=[False, True])
    wc = d[d['division_rank'] != 1].sort_values(order, ascending=[False, True])
    actual = pd.concat([dw, wc]).assign(actual_seed=range(1, 7))
    by_record = actual.sort_values(order, ascending=[False, True]).assign(record_seed=range(1, 7))
    for _, t in actual.merge(by_record[['team_id', 'record_seed']], on='team_id').iterrows():
        rows.append({'season': season, 'league': league, 'team': t['team'], 'record': t['record'],
                     'division winner': t['division_rank'] == 1,
                     'actual seed': t['actual_seed'], 'seed by record': t['record_seed']})
reseed = pd.DataFrame(rows)
reseed['spots'] = reseed['actual seed'] - reseed['seed by record']
reseed['bye'] = np.select([(reseed['seed by record'] <= 2) & (reseed['actual seed'] > 2),
                           (reseed['actual seed'] <= 2) & (reseed['seed by record'] > 2)],
                          ['gains bye', 'loses bye'], '')
reseed[reseed['spots'] != 0].sort_values(['season', 'league', 'seed by record'])

,season,league,team,record,division winner,actual seed,seed by record,spots,bye
9,2022,NL,NYM,101–61,False,4,3,1,
8,2022,NL,STL,93–69,True,3,4,-1,
15,2023,AL,TB,99–63,False,4,2,2,gains bye
13,2023,AL,HOU,90–72,True,2,3,-1,loses bye
16,2023,AL,TEX,90–72,False,5,4,1,
17,2023,AL,TOR,89–73,False,6,5,1,
14,2023,AL,MIN,87–75,True,3,6,-3,
27,2024,AL,BAL,91–71,False,4,3,1,
26,2024,AL,HOU,88–73,True,3,4,-1,
33,2024,NL,SD,93–69,False,4,3,1,


In [12]:
changed = reseed[reseed['spots'] != 0].groupby(['season', 'league']).ngroups
print(f"Seeds change in {changed} of {reseed.groupby(['season', 'league']).ngroups} league-seasons")
print('Would gain a bye:', reseed.query("bye == 'gains bye'")[['season', 'league', 'team', 'record']].to_dict('records'))
print('Would lose a bye:', reseed.query("bye == 'loses bye'")[['season', 'league', 'team', 'record']].to_dict('records'))
print('Total seed spots moved up:', int(reseed['spots'].clip(lower=0).sum()))

Seeds change in 6 of 10 league-seasons
Would gain a bye: [{'season': 2023, 'league': 'AL', 'team': 'TB', 'record': '99–63'}, {'season': 2025, 'league': 'AL', 'team': 'NYY', 'record': '94–68'}, {'season': 2026, 'league': 'AL', 'team': 'NYY', 'record': '93–68'}]
Would lose a bye: [{'season': 2023, 'league': 'AL', 'team': 'HOU', 'record': '90–72'}, {'season': 2025, 'league': 'AL', 'team': 'SEA', 'record': '90–72'}, {'season': 2026, 'league': 'AL', 'team': 'CLE', 'record': '85–77'}]
Total seed spots moved up: 15


## 6. The rows in the published charts
Part 1 (missed the playoffs) uses the top-12 rule; Part 2 (seeding) compares each Wild Card with the worst division winner in its league. Schedule-adjusted differences under 1 game are dropped since those records are estimates.

In [13]:
def lens_fields(t, lens):
    if lens == 'actual':
        return t['pct'], f"{t['wins']}–{t['losses']}"
    w = round(t['adj_pct'] * 162)
    return t['adj_pct'], f"~{w}–{162 - w}"

def diff(a, b, lens):
    return (a['net'] - b['net']) / 2 if lens == 'actual' else round((a['adj_pct'] - b['adj_pct']) * 162, 1)

graphic_rows = {}
for lens, key in (('actual', 'pct'), ('adjusted', 'adj_pct')):
    min_diff = 0 if lens == 'actual' else 1.0
    rows = []
    for season, d in twelve.groupby('season'):
        # The worst team that got in; equal records go by MLB's tiebreaker order (higher league_rank = lower)
        worst = d[d['made_playoffs']].sort_values([key, 'league_rank'], ascending=[True, False]).iloc[0]
        # Only teams that would have been in MLB's top 12 by record (they'd actually have gotten a spot)
        top12 = set(d.sort_values([key, 'league_rank'], ascending=[False, True])['team_id'].head(12))
        missed_in = d[~d['made_playoffs'] & d['team_id'].isin(top12)].sort_values(key, ascending=False)
        for _, up in missed_in.iterrows():
            rows.append((season, 'both', up, worst))
    graphic_rows[('cutoff', lens)] = rows
    rows = []
    for (season, league), d in twelve.groupby(['season', 'league']):
        dws = d[d['division_rank'] == 1]
        worst = dws.loc[dws[key].idxmin()]
        for _, w in d[d['made_playoffs'] & (d['division_rank'] != 1)].iterrows():
            rows.append((season, league, w, worst))
    graphic_rows[('wildcard', lens)] = rows

    for name in ('cutoff', 'wildcard'):
        out = []
        for season, league, up, low in graphic_rows[(name, lens)]:
            up_pct, up_rec = lens_fields(up, lens)
            low_pct, low_rec = lens_fields(low, lens)
            g = diff(up, low, lens)
            if g > 0 and g >= min_diff:
                out.append({'season': season, 'league': league, 'up_team': up['team'], 'up_record': up_rec,
                            'low_team': low['team'], 'low_record': low_rec, 'win difference': g})
        table = pd.DataFrame(out).sort_values('win difference', ascending=False)
        print(name, lens); display(table)

cutoff actual


,season,league,up_team,up_record,low_team,low_record,win difference
3,2026,both,AZ,86–76,HOU,81–81,5.0
0,2023,both,SEA,88–74,AZ,84–78,4.0
2,2025,both,HOU,87–75,CIN,83–79,4.0
1,2024,both,AZ,89–73,DET,86–76,3.0


wildcard actual


,season,league,up_team,up_record,low_team,low_record,win difference
7,2026,AL,NYY,93–68,HOU,81–81,12.5
1,2023,AL,TB,99–63,MIN,87–75,12.0
0,2022,NL,NYM,101–61,STL,93–69,8.0
5,2025,AL,NYY,94–68,CLE,88–74,6.0
8,2026,AL,BOS,87–75,HOU,81–81,6.0
3,2023,AL,TEX,90–72,MIN,87–75,3.0
9,2026,AL,CWS,84–78,HOU,81–81,3.0
4,2024,AL,BAL,91–71,HOU,88–73,2.5
2,2023,AL,TOR,89–73,MIN,87–75,2.0
6,2025,AL,BOS,89–73,CLE,88–74,1.0


cutoff adjusted


,season,league,up_team,up_record,low_team,low_record,win difference
3,2026,both,AZ,~86–76,HOU,~78–84,8.8
1,2024,both,AZ,~91–71,DET,~84–78,6.9
4,2026,both,PIT,~84–78,HOU,~78–84,6.2
2,2025,both,HOU,~86–76,CIN,~84–78,2.9
0,2023,both,SEA,~86–76,MIN,~84–78,2.4


wildcard adjusted


,season,league,up_team,up_record,low_team,low_record,win difference
8,2026,AL,NYY,~94–68,HOU,~78–84,15.9
2,2023,AL,TB,~99–63,MIN,~84–78,15.4
1,2022,NL,NYM,~101–61,STL,~90–72,10.9
9,2026,AL,BOS,~87–75,HOU,~78–84,9.8
6,2025,AL,NYY,~95–67,CLE,~87–75,7.5
3,2023,AL,TOR,~90–72,MIN,~84–78,6.1
10,2026,AL,CWS,~83–79,HOU,~78–84,4.9
0,2022,AL,TOR,~94–68,CLE,~89–73,4.6
4,2023,AL,TEX,~88–74,MIN,~84–78,4.4
5,2024,AL,BAL,~90–72,HOU,~86–76,3.9
